# Figure 11 -- load balance on a clustered distribution

Loads `bench/results/multigpu/load_balance.json`, produced by `bench/multigpu/load_balance.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("multigpu/load_balance.json")
cfg, recs = art["config"], art["data"]["records"]

ok = [r for r in recs if r.get("valid")]
if not ok:
    raise SystemExit("load_balance.json has no valid points")

dists = [d for d in cfg["distributions"] if any(r["distribution"] == d for r in ok)]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Left: pair work per device. A space-filling-curve split balances *particles* by
# construction, so a uniform cube looks perfect whatever the partition does. The
# clustered case is the one that can disagree, because the device holding the
# dense centre is handed more pair work than the ones holding the outskirts.
width = 0.8 / max(1, len(dists))
for di, dist in enumerate(dists):
    sel = [r for r in ok if r["distribution"] == dist]
    r = max(sel, key=lambda r: r["ndev"])
    work = r["per_device_work_total"]
    xs = [i + (di - (len(dists) - 1) / 2) * width for i in range(len(work))]
    axes[0].bar(xs, work, width=width, label=dist,
                color=style.entity_color(dist, di), edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("device")
axes[0].set_ylabel("interaction pairs")
axes[0].set_xticks(range(len(work)))
axes[0].set_title("pair work per device", fontsize=8)
style.finish(axes[0], legend=True, legend_kwargs={"loc": "best", "fontsize": 6.2})

# Right: imbalance vs device count. 1.0 is perfect; the ratio is the busiest
# device over the mean, which is what sets the step time -- every other device
# waits for it.
for di, dist in enumerate(dists):
    sel = sorted((r for r in ok if r["distribution"] == dist), key=lambda r: r["ndev"])
    axes[1].plot([r["ndev"] for r in sel], [r["work_imbalance"] for r in sel],
                 marker=style.MARKERS[di % len(style.MARKERS)],
                 color=style.entity_color(dist, di), label=dist)
axes[1].axhline(1.0, linestyle=":", linewidth=0.8, color="0.5")
axes[1].set_xlabel("devices")
axes[1].set_ylabel("busiest device / mean")
axes[1].set_title("work imbalance", fontsize=8)
style.finish(axes[1], legend=True, legend_kwargs={"loc": "best", "fontsize": 6.2})

style.annotate_config(
    axes[0],
    jsonio.config_caption(cfg, ["order", "leaf_size", "precision", "device"])
    + f"\n{cfg.get('per_device_n_fixed')} particles/device",
)
style.save(fig, FIG_DIR / "fig11_load_balance.pdf")

for r in sorted(ok, key=lambda r: (r["distribution"], r["ndev"])):
    print(f"{r['distribution']:<9s} ndev={r['ndev']:<3d} "
          f"imbalance={r['work_imbalance']:.4f} work={r['per_device_work_total']}")


## Caption

Per-device pair work for a uniform and a centrally concentrated (Plummer)
distribution, and the resulting imbalance against device count. The imbalance is
the busiest device's share divided by the mean, which is the quantity that sets
the step time because every other device waits on it. A uniform cube is balanced
by construction under any reasonable partition, so it is shown as the control;
the clustered case is the one in which balancing particle counts and balancing
work come apart.